# Isolating a Fortran Child Procedure

This notebook demonstrates how to isolate a Fortran procedure (subroutine or function) from its original module and generate a self-contained version that can be compiled and executed independently.

## Isolation Workflow

The isolation process follows these main steps:

1. **Identify the procedure**
   - Retrieve the procedure tree.
   - Determine whether it is a subroutine or a function.

2. **Find procedure variables**
   - Extract dummy arguments.
   - Identify local variables.
   - Identify global variables used by the procedure.
   - Detect variables modified inside the procedure.

3. **Filter and resolve global variables**
   - Remove variables corresponding to nested procedure calls.
   - Resolve additional global variables referenced through array/shape variables.
   - Extract global variable declarations.

4. **Extract array information**
   - Collect array dimensions and bounds.
   - Store the array shape information required by the isolated procedure.

5. **Handle nested procedures**
   - Identify procedures called from the target procedure.
   - Collect the corresponding procedure trees.

6. **Prepare the isolated procedure**
   - Create a copy of the procedure tree.
   - Remove unnecessary I/O statements.
   - Initialize the `Isolator` and prepare the working directories.

7. **Generate the isolated module**
   - Organize global declarations and code components.
   - Generate the global module containing the required declarations.
   - Generate the main program used to call the isolated procedure.

8. **Generate the procedure call**
   - Build the appropriate `CALL` statement for a subroutine.
   - Build an assignment statement for a function result.

9. **Generate and write Fortran code**
   - Update the main program.
   - Write the generated module and procedure code to the target directory.

10. **Compile and run**
    - Compile the generated Fortran code.
    - Execute the generated program.
    - Verify that compilation and execution complete successfully.

## Expected Result

At the end of the workflow, the selected procedure and all required dependencies are isolated into a dedicated directory. The generated Fortran code can then be compiled and tested independently from the original source module.


In [1]:
# Notebook Setup

%load_ext autoreload
%autoreload 2

import os

# Change to the Fgpt directory
%cd /home/kardaneh/Fgpt

# Fortran Parser
from fparser.two import Fortran2003 as F23

# Fgpt Imports
from fgpt.core.frontend import Processor
from fgpt.core.frontend import Extractor
from fgpt.core.common import Logger

# Initialize Fgpt
logger = Logger()
processor = Processor(logger=logger)

/home/kardaneh/Fgpt/.venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/home/kardaneh/Fgpt


In [2]:
# Fortran Module
file_namef90 = "/home/kardaneh/Fgpt/examples/example_01.f90"
file_name_org = file_namef90.replace(".f90", "_org.fgpt")
module = file_name_org if os.path.exists(file_name_org) else file_namef90

module_dir = "/home/kardaneh/Fgpt/examples/"

# Parse Fortran File
tree = processor.parse_fortran_file(module)

# Initialize Extractor
cls = Extractor(module_dir, tree, logger)

[INFO] Successfully parsed file: /home/kardaneh/Fgpt/examples/example_01.f90

[INFO] Normalizing names in parsed file: /home/kardaneh/Fgpt/examples/example_01.f90

[INFO] Successfully normalized all names in the module

In [3]:
# Find Subroutines

module_name = "example_01"
cls.find_subroutines(module_name)

# Inspect Extracted Subroutines and Calls
logger.info(f"Subroutines: {list(cls.subroutines.keys())}")
logger.info(f"Call within subroutines: {list(cls.call_within_sub.keys())}")
logger.info(
    f"Calls in 'test_procedures': "
    f"{list(cls.call_within_sub['test_procedures'].keys())}"
)

# Define a procedure Relationship
child_procedure = "set_data"
parent_procedure = "test_procedures"

[WARNING] ⚠ Suffix found in function 'compute_average': RESULT(avg)

[INFO] Function compute_average is called as compute_average(data_array, n, variance, stddev) inside of procedure 
test_procedures.

[INFO] Subroutines: ['set_data', 'process_data', 'display_data', 'dump_to_file', 'reset_data', 'scale_data', 
'compute_average', 'test_procedures']

[INFO] Call within subroutines: ['test_procedures']

[INFO] Calls in 'test_procedures': ['compute_average', 'set_data', 'display_data', 'process_data', 'scale_data', 
'dump_to_file', 'reset_data']

In [4]:
# Get Call Statements
call_statements = cls.call_within_sub.get(parent_procedure, {}).get(child_procedure, [])

# Inspect Call Sites
for i, call_stmt in enumerate(call_statements):
    logger.info(f"  Call site {i + 1}: {call_stmt.tostr()}")

[INFO]   Call site 1: CALL set_data(data_array, n, 0.0, 1.5)

[INFO]   Call site 2: CALL set_data(data_array, n, 1.0, 1.5)

In [5]:
# Get Procedure Tree
procedure_tree = cls.subroutines[child_procedure]
logger.info(f"Procedure tree:\n{procedure_tree}")

# Determine Procedure Type
if isinstance(procedure_tree, F23.Subroutine_Subprogram):
    procedure_type = "subroutine"

elif isinstance(procedure_tree, F23.Function_Subprogram):
    procedure_type = "function"

else:
    raise ValueError(
        f"Unknown procedure type: {type(procedure_tree)}"
    )

[INFO] Procedure tree:
SUBROUTINE set_data(arr, size, start, step)
  IMPLICIT NONE
  INTEGER, INTENT(IN) :: size
  REAL, INTENT(OUT) :: arr(size)
  REAL, INTENT(IN) :: start
  REAL, INTENT(IN) :: step
  INTEGER :: i

  DO i = 1, size
    arr(i) = start + REAL(i - 1) * step
  END DO

  PRINT *, "Data set successfully! start=", start, " step=", step
END SUBROUTINE set_data

In [6]:
# Find Variables: dummy arguments, local variables, and global variables
cls.find_variables(
    procedure_tree,
    child_procedure,
    parent_procedure,
)

In [7]:
subroutine_key = child_procedure
logger.info(f"declared variables: {cls.var_declared[subroutine_key]}")
logger.info(f"dummy args list: {cls.dummy_arg_list[subroutine_key]}")

for i, item in enumerate(cls.var_dummy[subroutine_key]):
    logger.info(f"The dummy argument {i} is: {item}")

logger.info(f" Global variables: {cls.var_global[subroutine_key]}")

for i, item in enumerate(cls.var_local[subroutine_key]):
    logger.info(f"The local variable {i} is: {item}")

[INFO] declared variables: {'arr', 'size', 'i', 'start', 'step'}

[INFO] dummy args list: ['arr', 'size', 'start', 'step']

[INFO] The dummy argument 0 is: REAL, INTENT(OUT) :: arr(size)

[INFO] The dummy argument 1 is: INTEGER, INTENT(IN) :: size

[INFO] The dummy argument 2 is: REAL, INTENT(IN) :: start

[INFO] The dummy argument 3 is: REAL, INTENT(IN) :: step

[INFO]  Global variables: []

[INFO] The local variable 0 is: INTEGER :: i

In [8]:
# Filter Global Variables
calls = cls.call_within_sub[child_procedure]

cls.var_global[child_procedure] = [
    name
    for name in cls.var_global[child_procedure]
    if name.tostr() not in calls
]

logger.info(f"Global variables: {cls.var_global[child_procedure]}")

# Find Global Variables
if cls.var_global[child_procedure]:
    logger.info(
        f"Global variables to find: {cls.var_global[child_procedure]}"
    )

    cls.find_global_variables(
        module_dir,
        tree,
        cls.var_global[child_procedure],
        child_procedure,
    )

[INFO] Global variables: []

In [9]:
# Process Dummy Variables
if cls.var_dummy[child_procedure]:
    logger.info(
        f"Dummy variables to process:\n {cls.var_dummy[child_procedure]}"
    )

    cls.process_declaration_variables(
        cls.var_dummy[child_procedure],
        child_procedure,
    )

logger.info(f"scaler names list: {cls.scalar_variables[subroutine_key]}")
logger.info(f"shape names list: {cls.shapes_variables[subroutine_key]}")

[INFO] Dummy variables to process:
 [Type_Declaration_Stmt(Intrinsic_Type_Spec('REAL', None), Attr_Spec_List(',', (Intent_Attr_Spec('INTENT', 
Intent_Spec('OUT')),)), Entity_Decl_List(',', (Entity_Decl(Name('arr'), Explicit_Shape_Spec_List(',', 
(Explicit_Shape_Spec(None, Name('size')),)), None, None),))), Type_Declaration_Stmt(Intrinsic_Type_Spec('INTEGER', 
None), Attr_Spec_List(',', (Intent_Attr_Spec('INTENT', Intent_Spec('IN')),)), Entity_Decl_List(',', 
(Entity_Decl(Name('size'), None, None, None),))), Type_Declaration_Stmt(Intrinsic_Type_Spec('REAL', None), 
Attr_Spec_List(',', (Intent_Attr_Spec('INTENT', Intent_Spec('IN')),)), Entity_Decl_List(',', 
(Entity_Decl(Name('start'), None, None, None),))), Type_Declaration_Stmt(Intrinsic_Type_Spec('REAL', None), 
Attr_Spec_List(',', (Intent_Attr_Spec('INTENT', Intent_Spec('IN')),)), Entity_Decl_List(',', 
(Entity_Decl(Name('step'), None, None, None),)))]

[INFO] scaler names list: [Name('size'), Name('start'), Name('step')]

[INFO] shape names list: [Name('size')]

In [10]:
# Process Global Variable Declarations
if cls.dec_global[child_procedure]:
    logger.info(
        f"Global variable declarations: {cls.dec_global[child_procedure]}"
    )

    for key in cls.dec_global[child_procedure].keys():
        cls.process_declaration_variables(
            cls.dec_global[child_procedure][key],
            child_procedure,
        )

In [11]:
# Find Global Variables from Shape Variables
scalar_names = {
    var.tostr()
    for var in cls.scalar_variables[child_procedure]
}

global_names = {
    var.tostr()
    for var in cls.var_global[child_procedure]
}

shape_to_search = [
    var
    for var in cls.shapes_variables[child_procedure]
    if var.tostr() not in scalar_names
    and var.tostr() not in global_names
]

logger.info(f"Shape variables to search: {shape_to_search}")

if shape_to_search:
    cls.find_global_variables(
        module_dir,
        tree,
        shape_to_search,
        child_procedure,
    )

    cls.var_global[child_procedure].extend(shape_to_search)

    logger.info(
        f"Global variables after shape search: "
        f"{cls.var_global[child_procedure]}"
    )

[INFO] Shape variables to search: []

In [12]:
# Extract All Array Information
cls.extract_all_array_info(
    cls.dec_global[child_procedure],
    cls.var_dummy[child_procedure],
    child_procedure,
)

logger.info(
    f"Array shape info for subroutine '{child_procedure}':"
)

for var_name, dims in cls.all_array_info[child_procedure].items():
    logger.info(f" - {var_name}:")

    for i, dim in enumerate(dims):
        logger.info(
            f"       Dim {i + 1}: "
            f"Start = {dim['dim_str']}, "
            f"End = {dim['dim_end']}"
        )

[INFO] Array shape info for subroutine 'set_data':

[INFO]  - arr:

[INFO]        Dim 1: Start = 1, End = size

In [13]:
# Check Nested Procedures
nested_procedures = cls.call_within_sub[child_procedure].keys()

if nested_procedures:
    logger.info(
        f"Found {len(nested_procedures)} nested procedure(s) "
        f"in '{child_procedure}':"
    )
else:
    logger.info(
        f"No nested procedures found in '{child_procedure}' - "
        f"proceeding to complete isolation"
    )

[INFO] No nested procedures found in 'set_data' - proceeding to complete isolation

In [14]:
# Extract Procedure Intent
cls.extract_intent(child_procedure, procedure_tree, calls)

logger.info(
    f"General usage dict: {cls.general_usage_dict[child_procedure]}"
)

# Clean Subroutine
cls.clean_subroutine(child_procedure, procedure_tree)

# Extract Local IN Variables
cls.extract_local_in_variables(child_procedure, procedure_tree)

logger.info(
    f"Local vars IN for procedure '{child_procedure}': "
    f"{cls.var_in_local[child_procedure]}"
)

# Extract Modified Variables
cls.extract_modified_variables(child_procedure, procedure_tree)

for entity_decl in cls.var_modif_info[child_procedure]:
    logger.info(
        f"{entity_decl}: "
        f"{cls.var_modif_info[child_procedure][entity_decl]}"
    )

# Organize Code Components
input_dict = cls.organize_code_components(
    child_procedure,
    cls.dec_global[child_procedure],
    openacc=False,
)

logger.info(
    f"Result:\n"
    f"  add to module: {input_dict['add_to_module']}\n"
    f"  add to routin: {input_dict['add_to_routin']}\n"
    f"  add to usestm: {input_dict['add_to_usestm']}\n"
    f"  add to dtyped: {input_dict['add_to_dtyped']}\n"
    f"  acc declare create: {input_dict['acc_declare_create']}\n"
    f"  acc declare copyin: {input_dict['acc_declare_copyin']}\n"
    f"  reads non allocatables: {input_dict['reads_non_allocatables']}\n"
    f"  reads allocatables: {input_dict['reads_allocatables']}\n"
    f"  write stmt: {input_dict['write_stmt']}"
)

[INFO] Induced INTENT for subroutine 'set_data':

[INFO]   'arr': 'OUT'

[INFO]   'size': 'IN'

[INFO]   'start': 'IN'

[INFO]   'step': 'IN'

...

[INFO] Successfully removed INTENT and SAVE attributes from statements

[INFO] Processing initialization completed!

[INFO] Processing initialization completed!

[INFO] Declarations and allocations processed successfully

[INFO] Result:
  add to module: []
  add to routin: []
  add to usestm: []
  add to dtyped: []
  acc declare create: []
  acc declare copyin: []
  reads non allocatables: []
  reads allocatables: []
  write stmt: []

In [15]:
# Prepare Subroutine Tree
subroutine_tree_cp = processor.parse_fortran_string(
    procedure_tree.tofortran()
)

processor.remove_io_statements(subroutine_tree_cp)

[INFO] Successfully parsed string!

In [16]:
# Initialize Isolator
from fgpt import Isolator

isolator = Isolator(
    rest_of_path="Fgpt/examples",
    target_module="example_01",
    work="/home/kardaneh",
    config_path="/home/kardaneh/Fgpt/template.yaml",
    openacc=False,
    tapenade=False,
    f2py=False,
    py2jx=False,
)

# Prepare Working Subroutines
isolator.working_subroutines[child_procedure] = subroutine_tree_cp

sub_trees = []

for sub_name in isolator.collect_all_subroutines(cls, child_procedure):
    sub_trees.append(
        isolator.working_subroutines[sub_name]
    )

# Configure Module Path
cls.module_path[isolator.target_module] = isolator.path_to_target

# Prepare Subroutine Directory
assert os.path.exists(isolator.processor.benchmark_dir), (
    "benchmark directory does not exist!"
)

sub_dir = os.path.join(
    isolator.processor.benchmark_dir,
    child_procedure,
)

os.makedirs(sub_dir, exist_ok=True)
logger.info(
    f"Working subroutines: {list(isolator.working_subroutines.keys())}"
)
logger.info(f"Subroutine directory: {sub_dir}")

╭────────────────────────────────── Fortran General purpose Transformer (Fgpt) ───────────────────────────────────╮
│ 🚀 Starting Module: Isolator                                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[INFO] Successfully parsed file: /home/kardaneh/Fgpt/examples/example_01.f90

[INFO] Normalizing names in parsed file: /home/kardaneh/Fgpt/examples/example_01.f90

[INFO] Successfully normalized all names in the module

[INFO] Working subroutines: ['set_data']

[INFO] Subroutine directory: /home/kardaneh/Fgpt/benchmark/set_data

In [17]:
# Create Target Module Directory
isolator.target_module_dir = os.path.join(
    "/home/kardaneh/Fgpt/examples/",
    isolator.target_module.split(".")[0],
)

isolator.create_target_directory()

# Create Subroutine Directory
subroutine_dir = os.path.join(
    isolator.target_module_dir,
    child_procedure,
)

os.makedirs(subroutine_dir, exist_ok=True)
logger.info(
    f"Created parent function directory: {subroutine_dir}"
)

# Load Fortran Code Templates
module_code_string = (
    isolator.code_templates["Fortran_global_module_template"]["general"]
)

main_code_string = (
    isolator.code_templates["Fortran_main_template"]["general"]
)

# Update Global Module
processor.update_global_module(
    module_code_string,
    main_code_string,
    input_dict,
    subroutine_dir,
    child_procedure,
    procedure_tree,
    sub_trees,
)

[INFO] Created parent function directory: /home/kardaneh/Fgpt/examples/example_01/set_data

[INFO] Successfully parsed module code

[INFO] Successfully parsed main program code

[INFO] Inserted I/O statements at beginning of Execution_Part in procedure: set_data

[INFO] Successfully updated the global module

[INFO] Successfully wrote code to file: /home/kardaneh/Fgpt/examples/example_01/set_data/module_global_set_data.f90

In [18]:
# Read Generated Global Module
file_path = os.path.join(
    subroutine_dir,
    f"module_global_{child_procedure}.f90",
)

with open(file_path, "r") as f:
    logger.info(f"\n{f.read()}")

[INFO] 
! Global module for set_data generated by fgpt
MODULE module_global_set_data
  IMPLICIT NONE
  INTEGER, PARAMETER :: i_std = 4
  INTEGER, PARAMETER :: r_std = 8
  INTEGER :: ier
  INTEGER(KIND = i_std) :: ic0
  INTEGER(KIND = i_std) :: ic
  REAL(KIND = r_std) :: icr
  REAL(KIND = r_std) :: start_time
  REAL(KIND = r_std) :: stop_time
  CONTAINS
  SUBROUTINE set_data(arr, size, start, step)
    IMPLICIT NONE
    INTEGER, INTENT(IN) :: size
    REAL, INTENT(OUT) :: arr(size)
    REAL, INTENT(IN) :: start
    REAL, INTENT(IN) :: step
    INTEGER :: i

    DO i = 1, size
      arr(i) = start + REAL(i - 1) * step
    END DO

    PRINT *, "Data set successfully! start=", start, " step=", step
  END SUBROUTINE set_data
END MODULE module_global_set_data

In [19]:
# Build Procedure Call Statement
arg_list = ", ".join(
    name
    for name in cls.dummy_arg_list[child_procedure]
)

if procedure_type == "subroutine":
    call_stmt_org = F23.Call_Stmt(
        f"CALL {child_procedure}({arg_list})"
    )

elif procedure_type == "function":
    call_stmt_org = F23.Assignment_Stmt(
        f"{cls.func_result[child_procedure]} = "
        f"{child_procedure}({arg_list})"
    )

else:
    raise ValueError(
        f"Unsupported procedure type: {procedure_type}"
    )

call_stmts = [call_stmt_org]

logger.info(f"Procedure call statement: {call_stmts}")

[INFO] Procedure call statement: [Call_Stmt(Name('set_data'), Actual_Arg_Spec_List(',', (Name('arr'), Name('size'),
Name('start'), Name('step'))))]

In [20]:
# Process Dummy Variable Declarations
from collections import defaultdict

dec_dummy = defaultdict(lambda: defaultdict(list))

for declaration in cls.var_dummy[child_procedure]:
    declaration_name, declaration_list = (
        isolator.processor.break_allocatable_declaration(
            declaration
        )
    )

    dec_dummy[child_procedure][declaration_name] = declaration_list

for declaration_name, declaration_list in dec_dummy[child_procedure].items():
    logger.info(
        f"Dummy variable declaration: {declaration_name}"
    )

    for item in declaration_list:
        logger.info(
            f"  - {item}"
        )

# Organize Code Components
input_dict = cls.organize_code_components(
    child_procedure,
    dec_dummy[child_procedure],
    openacc=False,
)

#logger.info(
#    f"Result:\n"
#    f"  add to module: {input_dict['add_to_module']}\n"
#    f"  add to routin: {input_dict['add_to_routin']}\n"
#    f"  add to usestm: {input_dict['add_to_usestm']}\n"
#    f"  add to dtyped: {input_dict['add_to_dtyped']}\n"
#    f"  acc declare create: {input_dict['acc_declare_create']}\n"
#    f"  acc declare copyin: {input_dict['acc_declare_copyin']}\n"
#    f"  reads non allocatables: {input_dict['reads_non_allocatables']}\n"
#    f"  reads allocatables: {input_dict['reads_allocatables']}\n"
#    f"  write stmt: {input_dict['write_stmt']}"
#)

[INFO] Allocatable declaration: REAL, ALLOCATABLE, DIMENSION(:) :: arr

[INFO] Allocate statement: ALLOCATE(arr(size))

[INFO] Successfully removed INTENT and SAVE attributes from statements

[INFO] Successfully removed INTENT and SAVE attributes from statements

[INFO] Successfully removed INTENT and SAVE attributes from statements

...

[INFO] Successfully parsed statement: READ(1363, IOSTAT = ier) step
IF (ier /= 0) THEN
  WRITE(*, *) 'Error reading from file for step. ', ' IOSTAT : ', ier
END IF

[INFO] Processing initialization completed!

[INFO] Successfully parsed statement: READ(1363, IOSTAT = ier) arr
IF (ier /= 0) THEN
  WRITE(*, *) 'Error reading from file for arr. ', ' IOSTAT : ', ier
END IF

[INFO] Processing initialization completed!

[INFO] Declarations and allocations processed successfully

In [21]:
# Update Main Program
processor.update_main_program(
    input_dict=input_dict,
    call_stmts=call_stmts,
    var_modif=cls.var_modif_info[child_procedure],
    subroutine_dir=subroutine_dir,
    subroutine_name=child_procedure,
    procedure_tree=procedure_tree,
    openacc=False,
    dummy_add_decl=None,
    error_flag=None,
    acc_data_copyin=None,
)

[INFO] Inserted I/O statements at beginning of Execution_Part in procedure: set_data

[INFO] Successfully parsed statement: CALL SYSTEM_CLOCK(ic0, icr, ic)
start_time = ic0 * 1.0 / icr

[INFO] Successfully parsed statement: CALL SYSTEM_CLOCK(ic0, icr, ic)
stop_time = ic0 * 1.0 / icr
WRITE(*, *) "Execution time : ", stop_time - start_time
OPEN(UNIT = 1363, FILE = '/home/kardaneh/Fgpt/benchmark/set_data/time.txt', STATUS = 'unknown', POSITION = 
'append')
WRITE(1363, *) stop_time - start_time
CLOSE(UNIT = 1363)

[INFO] Successfully parsed statement: OPEN(UNIT = 1363, FILE = '/home/kardaneh/Fgpt/benchmark/set_data/output.bin',
FORM = 'unformatted', STATUS = 'replace')
WRITE(1363) arr
CLOSE(UNIT = 1363)

[INFO] Successfully updated the main program

[INFO] Successfully wrote code to file: /home/kardaneh/Fgpt/examples/example_01/set_data/main_set_data.f90

In [22]:
# Write Module Tree to File
write_module_tree = procedure_tree.get_root()

path_to_save = os.path.join(
    isolator.module_dir_sp,
    f"{isolator.target_module}.f90",
)

isolator.processor.write_fortran_code_to_file(
    write_module_tree,
    path_to_save,
)

logger.info(f"Generated module written to: {path_to_save}")

[INFO] Successfully wrote code to file: /home/kardaneh/Fgpt/examples/example_01.f90

[INFO] Generated module written to: /home/kardaneh/Fgpt/examples/example_01.f90

In [23]:
# Compile and Run Generated Code
error_status = isolator.processor.compile_and_run(
    os.getcwd(),
    subroutine_dir,
    auto_diff=isolator.tapenade,
)

logger.info(f"Compilation and execution status: {error_status}")

assert error_status == 0, (
    "Error: Compilation failed or main_program not generated."
)

[INFO] Compiling and running in /home/kardaneh/Fgpt/examples/example_01/set_data...

Tapenade files detected (adjoint: , tangent: ) - enabling Tapenade runtime
rm -rf obj set_data mod /home/kardaneh/Fgpt/examples/example_01/set_data/set_data.txt
Cleaned build artifacts.
Tapenade files detected (adjoint: , tangent: ) - enabling Tapenade runtime
rm -rf obj set_data mod /home/kardaneh/Fgpt/examples/example_01/set_data/set_data.txt
Cleaned build artifacts.
mpif90 -Wall -g -O0 -Kieee -Ktrap=fp -Mbounds -traceback -r8 -i4 -I/data/ssivanes/modipsl_truck_opt/modeles/IOIPSL/inc -I/data/ssivanes/modipsl_truck_opt/modeles/XIOS/inc -I/data/ssivanes/modipsl_truck_opt/modeles/ORCHIDEE/inc -I/net/nfs/tools/u20/22.3/PrgEnv/nvhpc/linux-ubuntu20.04-x86_64/netcdf-c/4.7.4-nvhpc-21.9-qnkr5ug7ksav3drxwijn2dc4t75yzota/include -I/net/nfs/tools/u20/22.3/PrgEnv/nvhpc/linux-ubuntu20.04-x86_64/netcdf-fortran/4.5.3-nvhpc-21.9-ea7lcxvrs3byj4iwmp7eul7vfz5zhavo/include -c /home/kardaneh/Fgpt/examples/example_01/set_data/module_global_set_data.f90 -o obj/module_global_set_data.o -module mod >> /home/k

[INFO] Compilation process completed!

[INFO] Benchmark files exist. Now running the unit tests ...

 --- inside the main program ---
 --- inside the declaration_initialization routine for set_data ---
 --- inside the read_dummy routine for set_data ---
 Data set successfully! start=    1.000000000000000       step= 
    1.500000000000000     
 Execution time :    2.2810000000000000E-003


[INFO] Execution completed in /home/kardaneh/Fgpt/examples/example_01/set_data

[INFO] Compilation and execution status: 0